# 04 - Modelo de Churn

Este notebook mostra o processo did?tico de Machine Learning para churn. Ele compara modelos simples e explica por que o projeto usa Random Forest no pipeline oficial.

A entrega de produ??o local continua em:

- `src/train_model.py`
- `src/predict_churn.py`
- `modelo/model.pkl`

Este notebook ? uma evid?ncia de aprendizado e racioc?nio t?cnico para a banca.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DADOS_RAW = ROOT / "dados" / "raw"
DADOS_PROCESSED = ROOT / "dados" / "processed"
DADOS_OUTPUTS = ROOT / "dados" / "outputs"
print(f"Raiz do projeto: {ROOT}")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

In [ ]:
base = pd.read_csv(DADOS_PROCESSED / "base_modelagem.csv")
print(base.shape)
display(base.head())

## 1. Separa??o entre features e target

O target ? `churn_flag`. Removemos identificadores, nomes e datas para reduzir risco de vazamento e evitar que o modelo aprenda c?digos em vez de padr?es.

In [ ]:
target = "churn_flag"
colunas_remover = [
    "cliente_id", "customer_id_original", "nome", "data_cadastro", "ultimo_movimento", "primeira_transacao",
    "origem_dado", target
]
X = base.drop(columns=[c for c in colunas_remover if c in base.columns])
y = base[target].astype(int)

num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()

print(f"Features num?ricas: {len(num_cols)}")
print(f"Features categ?ricas: {cat_cols}")
print(f"Taxa baseline de churn: {y.mean():.2%}")

## 2. Train/test split com estratifica??o

Usamos `stratify=y` para manter a propor??o de churn nas bases de treino e teste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Treino:", X_train.shape, "Teste:", X_test.shape)
print("Churn treino:", y_train.mean().round(4), "Churn teste:", y_test.mean().round(4))

## 3. Pr?-processamento

Vari?veis num?ricas recebem imputa??o de mediana e escala. Vari?veis categ?ricas recebem imputa??o e one-hot encoding.

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)

## 4. Modelos comparados

- `LogisticRegression`: baseline simples e f?cil de explicar.
- `DecisionTreeClassifier`: modelo interpret?vel.
- `RandomForestClassifier`: modelo final mais robusto, usado no pipeline oficial.

In [ ]:
modelos = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1),
}

resultados = []
ajustados = {}

for nome, modelo in modelos.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", modelo)])
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    resultados.append({
        "modelo": nome,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1_score": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, prob),
    })
    ajustados[nome] = pipe

resultados_df = pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)
display(resultados_df)

## 5. Matriz de confus?o do modelo final

Para churn, o falso negativo ? especialmente sens?vel: ? o cliente que poderia sair, mas n?o entrou na fila de reten??o.

In [ ]:
modelo_final = ajustados["RandomForest"]
prob_final = modelo_final.predict_proba(X_test)[:, 1]
pred_final = (prob_final >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred_final)

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_title("Matriz de confus?o - Random Forest")
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_xticks([0, 1], labels=["N?o churn", "Churn"])
ax.set_yticks([0, 1], labels=["N?o churn", "Churn"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print(classification_report(y_test, pred_final, target_names=["N?o churn", "Churn"]))

## 6. Precision vs recall

- `precision` responde: entre os clientes alertados como risco, quantos realmente eram churn?
- `recall` responde: entre os clientes que eram churn, quantos o modelo conseguiu capturar?

Em reten??o, falso negativo importa porque significa deixar de acionar algu?m que poderia sair. Por isso o pipeline oficial compara thresholds como 0.30, 0.40 e 0.50 para equilibrar captura e qualidade dos alertas.

In [ ]:
thresholds = [0.30, 0.40, 0.50]
threshold_df = []
for th in thresholds:
    pred = (prob_final >= th).astype(int)
    threshold_df.append({
        "threshold": th,
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1_score": f1_score(y_test, pred, zero_division=0),
        "alert_rate": pred.mean(),
    })
threshold_df = pd.DataFrame(threshold_df)
display(threshold_df)

plt.figure(figsize=(7, 4))
plt.plot(threshold_df["threshold"], threshold_df["precision"], marker="o", label="precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], marker="o", label="recall")
plt.title("Trade-off precision vs recall")
plt.xlabel("Threshold")
plt.ylabel("M?trica")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Feature importance

A import?ncia de vari?veis ajuda a explicar quais sinais mais influenciaram o modelo final. Para a banca, isso conecta Machine Learning com interpreta??o de neg?cio.

In [ ]:
feature_names = modelo_final.named_steps["preprocess"].get_feature_names_out()
importances = modelo_final.named_steps["model"].feature_importances_
fi = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False).head(15)
display(fi)

fi.sort_values("importance").plot(kind="barh", x="feature", y="importance", figsize=(8, 6), color="#20c7df", legend=False)
plt.title("Top feature importance - Random Forest")
plt.tight_layout()
plt.show()

## 8. Predi??es em produ??o local

Este notebook demonstra o processo. A gera??o oficial de `dados/outputs/predicoes_churn.csv` acontece no pipeline:

```powershell
python -m src.pipeline
```

ou pela infer?ncia separada:

```powershell
python -m src.predict_churn
```

O modelo final salvo em produ??o local fica em `modelo/model.pkl`.